# Lesson 1.2: How LLMs Actually Work

**The mental model you need before you ship anything to production.**

30-45 min. Concept-heavy. Code: tokenize text and watch the model split it.

## Architect Level

- [ ] Junior (1-2 yrs) -- implement and run
- [ ] Mid (3-5 yrs) -- extend with monitoring
- [ ] Senior (6-10 yrs) -- add multi-tenancy
- [ ] Staff (10+ yrs) -- design the SLA, capacity model, incident playbook

---

## Concept (5 min)

A Large Language Model is **two things stacked**:

1. **A tokenizer** -- chops text into integer IDs the model understands.
2. **A transformer** -- a stack of matrix multiplications + attention that predicts the next token.

Three mental models:
- **Context window = RAM.** When full, older context falls out.
- **Temperature = dice roll.** 0 = greedy, 1 = creative.
- **System prompt = configuration.** Sets persona, not task.

## Build It: Starter Code

Run the cells below. The first cell has TODOs; the second cell is the solution.

In [ ]:
import tiktoken

def count_tokens(text: str, encoding_name: str = "cl100k_base") -> int:
    """TODO: Return the number of tokens in `text` for the given encoding."""
    pass

def show_tokens(text: str, encoding_name: str = "cl100k_base") -> tuple:
    """TODO: Return (token_ids, decoded_tokens) for the given text."""
    pass

def render_token_bar(token_count: int, max_count: int, width: int = 50) -> str:
    """TODO: Return '#' chars proportional to token_count / max_count."""
    pass

def compare_encodings(text: str) -> None:
    """TODO: Print token counts under 3+ different encodings."""
    pass

In [ ]:
# SOLUTION -- Complete version
def count_tokens(text: str, encoding_name: str = "cl100k_base") -> int:
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(text))

def show_tokens(text: str, encoding_name: str = "cl100k_base") -> tuple:
    encoding = tiktoken.get_encoding(encoding_name)
    ids = encoding.encode(text)
    tokens = [encoding.decode([t]) for t in ids]
    return ids, tokens

def render_token_bar(token_count: int, max_count: int, width: int = 50) -> str:
    if max_count <= 0:
        return ""
    bar_len = int((token_count / max_count) * width)
    return "#" * bar_len

def compare_encodings(text: str) -> None:
    encodings_to_test = ["cl100k_base", "p50k_base", "o200k_base"]
    print(f"\n  Comparing encodings for: {text!r}")
    print(f"  {'Encoding':<20} {'Tokens':>8}")
    for enc_name in encodings_to_test:
        try:
            n = count_tokens(text, enc_name)
            print(f"  {enc_name:<20} {n:>8}")
        except KeyError:
            print(f"  {enc_name:<20} (not available)")

## Demo 1: Tokenize 6 different strings

In [ ]:
examples = [
    ("Short sentence", "Hello, world!"),
    ("Long sentence",  "Large language models predict the next token in a sequence."),
    ("Code snippet",   "def fibonacci(n):\n    return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)"),
    ("URL",            "https://platform.openai.com/docs/api-reference/chat/create"),
    ("Number",         "3.14159265358979323846264338327950288419716939937510"),
    ("Non-English",    "Bonjour le monde! \u3053\u3093\u306b\u3061\u306f\u4e16\u754c\u3002"),
]

for label, text in examples:
    ids, tokens = show_tokens(text)
    print(f"\n[{label}]")
    print(f"  Text:   {text[:60]!r}{'...' if len(text) > 60 else ''}")
    print(f"  Tokens: {len(ids)}")
    print(f"  IDs:    {ids[:8]}{'...' if len(ids) > 8 else ''}")
    print(f"  Pieces: {tokens}")

## Demo 2: Visualize with a bar chart

In [ ]:
all_counts = [count_tokens(text) for _, text in examples]
max_count = max(all_counts)

print("Token count comparison (1 # = 1 token):\n")
for (label, _), count in zip(examples, all_counts):
    bar = render_token_bar(count, max_count, width=50)
    print(f"  {count:3d} {bar}  {label}")

## Demo 3: Cost estimation

In [ ]:
PRICING = {
    "gpt-4o":            {"input": 5.00,  "output": 15.00},
    "gpt-4o-mini":       {"input": 0.15,  "output": 0.60},
    "claude-3.5-sonnet": {"input": 3.00,  "output": 15.00},
    "claude-3.5-haiku":  {"input": 0.80,  "output": 4.00},
}

print("Cost per 1M input tokens (2026):\n")
for model, p in PRICING.items():
    print(f"  {model:<22} ${p['input']:.2f}")

print("\nA 2K input + 500 output request:")
for model, p in PRICING.items():
    cost = (2000 / 1_000_000) * p['input'] + (500 / 1_000_000) * p['output']
    print(f"  {model:<22} ${cost:.6f} per request")

## Architect Notes

### Trade-offs

| Tokenizer     | Pros                          | Cons                                | Pick when                |
|---------------|-------------------------------|-------------------------------------|--------------------------|
| tiktoken      | Same as GPT-4, accurate costs | Locked to OpenAI encoding           | You're using OpenAI      |
| HF tokenizer  | Works with any HF model       | Different IDs than OpenAI           | Mixing model families    |
| Character     | Trivial, no dependencies      | 4-10x more tokens, cost explodes    | Never (only toy demos)   |
| SentencePiece | Open-source friendly          | Each model has its own              | Open-source models       |

### Capacity Model

Tokenization is **CPU-bound, not network-bound**: ~50K tokens/sec per core. It never bottlenecks production. **Where it DOES bite:** storing tokens (1M tokens = 4 MB). Pre-tokenize at ingest, not at query.

### Cost Model (per 1M tokens, 2026)

| Model              | Input | Output |
|--------------------|-------|--------|
| GPT-4o             | $5    | $15    |
| GPT-4o-mini        | $0.15 | $0.60  |
| Claude 3.5 Sonnet  | $3    | $15    |
| Claude 3.5 Haiku   | $0.80 | $4     |
| Gemini Pro 1.5     | $1.25 | $5     |
| Llama 3 70B (self) | $0.10 | $0.10  |

### Production Checklist

- [ ] Tokenizer loaded once at app startup, not per request
- [ ] Every LLM call logs prompt_tokens, completion_tokens, total_tokens
- [ ] Cost computed at request time using current model pricing
- [ ] Long contexts (>10K tokens) flagged in logs
- [ ] Per-user token budgets enforced at the API layer
- [ ] Context size checked BEFORE the LLM call
- [ ] Cost dashboards exist by model, user, and time window

## Reflect (10 min)

1. What did I build?  A tokenizer visualizer. What did you learn about how text becomes numbers?
2. What was hard?  Was the ASCII bar chart? The encoding comparison? The cost math?
3. What would I change at 10x scale?  If you had to tokenize 10M documents/day, what would you do?
4. What's tomorrow's lab?  Lesson 1.3: The Modern AI Stack. Compare 3 LLM APIs on the same prompt.

---

**References**

- [tiktoken on GitHub](https://github.com/openai/tiktoken)
- [Hugging Face tokenizers summary](https://huggingface.co/docs/transformers/tokenizer_summary)
- [OpenAI tokenizer playground](https://platform.openai.com/tokenizer)
- [Codebook section 1.1-1.5](../../workbooks/ai-engineer-codebook.md)
- [Paired codebook exercises](../../workbooks/exercises/section-1-llm-apis-exercises.md)